# parameter-subclass-of-tensor — faded example 3: complete freeze-by-prefix match

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `parameter-subclass-of-tensor`. The last cell reports your progress on the `Backprop: Parameter subclasses Tensor` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Parameter subclasses Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`parameter-subclass-of-tensor`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "parameter-subclass-of-tensor"
DD_SUBTOPIC = "Backprop: Parameter subclasses Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Freezing by prefix sets `requires_grad=False` on Parameters whose dotted name starts with the prefix, using a literal `str.startswith`. The type tag is untouched, so the params remain Parameters.

## Faded exercise 3

Complete `freeze` so it matches parameter names by prefix using startswith. Fill in the prefix-match condition.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class Parameter(MiniTensor):
    def __init__(self, array, requires_grad=True):
        super().__init__(array, requires_grad=requires_grad)

class Model:
    def __init__(self):
        self._params = [('enc.w', Parameter([1.0])), ('head.w', Parameter([2.0]))]
    def parameters(self):
        for name, p in self._params:
            yield name, p

def freeze(module, prefix):
    for name, p in module.parameters():
        match = None  # TODO: fill in this step — read the prompt cell above
        if match:
            p.requires_grad = False

m = Model()
freeze(m, 'enc.')
print([(n, p.requires_grad) for n, p in m.parameters()])


def _test():
    m = Model()
    freeze(m, 'enc.')
    rg = dict((n, p.requires_grad) for n, p in m.parameters())
    assert rg['enc.w'] is False   # frozen
    assert rg['head.w'] is True   # untouched
    # type tag preserved
    for _, p in m.parameters():
        assert isinstance(p, Parameter)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class Parameter(MiniTensor):
    def __init__(self, array, requires_grad=True):
        super().__init__(array, requires_grad=requires_grad)

class Model:
    def __init__(self):
        self._params = [('enc.w', Parameter([1.0])), ('head.w', Parameter([2.0]))]
    def parameters(self):
        for name, p in self._params:
            yield name, p

def freeze(module, prefix):
    for name, p in module.parameters():
        match = name.startswith(prefix)
        if match:
            p.requires_grad = False

m = Model()
freeze(m, 'enc.')
print([(n, p.requires_grad) for n, p in m.parameters()])
```
</details>